# HydroSeason 0.1.1 hybrid user routine

This notebook exercises the release candidate through public Python and CLI APIs. It runs offline by default using the committed Fitzroy extent CSV and AOI (about 1–3 minutes). Optional live DEA single-AOI and two-row batch paths require network access and can take 10–30 minutes.

Install `hydroseason[all,dev]` in this kernel environment. If the preflight reports the wrong version or missing APIs, run `%pip install -e "..[all,dev]"` from `notebooks/` (or `%pip install -e ".[all,dev]"` from the repository root), then restart the kernel.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

import pandas as pd
from IPython.display import FileLink, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "case_studies").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Start this notebook from the repository root or notebooks directory.")

RUN_LIVE_DEA = os.environ.get("HYDROSEASON_RUN_LIVE_DEA", "0") == "1"
OUTPUT_DIR = Path(
    os.environ.get(
        "HYDROSEASON_NOTEBOOK_OUTPUT",
        str(PROJECT_ROOT / "notebooks/output/05_0_1_improvements"),
    )
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CASE_STUDY_CSV = PROJECT_ROOT / "case_studies/data/extent/fitzroy_river_wa_30m.csv"
FITZROY_AOI = PROJECT_ROOT / "data/fitzroy_catchment.geojson"
SMALL_AOI = PROJECT_ROOT / "data/fitzroy_kimberley_aoi.geojson"
SECOND_AOI = PROJECT_ROOT / "data/Gilbert_river_buffer.geojson"


In [ ]:
import hydroseason

required_api = {
    "run_hydroseason",
    "run_hydroseason_many",
    "analyze_catchment",
    "load_aoi",
    "HistoricalMaskCoverageWarning",
    "HistoricalMaskRefreshedWarning",
    "probe_wo_statistics_coverage",
}
missing = sorted(name for name in required_api if not hasattr(hydroseason, name))
project = tomllib.loads((PROJECT_ROOT / "pyproject.toml").read_text(encoding="utf-8"))
expected_version = project["project"]["version"]
if hydroseason.__version__ != expected_version or missing:
    raise RuntimeError(
        f"This kernel is not using the HydroSeason {expected_version} candidate. "
        f"Imported {hydroseason.__version__} from {hydroseason.__file__}; "
        f"missing APIs: {missing}. Run %pip install -e \"..[all,dev]\" from "
        "notebooks/ (or %pip install -e \".[all,dev]\" from the repo root), "
        "then restart the kernel."
    )
if not (sys.version_info[:2] >= (3, 10) and sys.version_info[:2] < (3, 14)):
    raise RuntimeError(f"Python {sys.version.split()[0]} is outside >=3.10,<3.14.")
print(hydroseason.__version__, hydroseason.__file__)


In [ ]:
import geopandas as gpd

from hydroseason import analyze_catchment, load_aoi, run_hydroseason, run_hydroseason_many


In [ ]:
offline_result = run_hydroseason(
    CASE_STUDY_CSV,
    output_dir=OUTPUT_DIR / "offline-fitzroy",
    aoi=FITZROY_AOI,
    aoi_name="Fitzroy River (WA)",
    analysis_options={"n_bootstrap": 999, "random_state": 0},
    progress=True,
    show_map="auto",
)
assert offline_result.source_kind == "extent_csv"
assert offline_result.aoi_context is not None
assert offline_result.artifacts.html.exists()


In [ ]:
timing = offline_result.analysis.regime
timing_summary = pd.DataFrame(
    [
        {
            "regime": timing.regime,
            "route": offline_result.analysis.route,
            "amplitude SNR": timing.amplitude_snr,
            "peak R": timing.peak_timing_concentration,
            "peak CI low": timing.peak_timing_concentration_ci_low,
            "peak CI high": timing.peak_timing_concentration_ci_high,
            "peak Kuiper p": timing.peak_timing_uniformity_p,
            "trough R": timing.trough_timing_concentration,
            "trough CI low": timing.trough_timing_concentration_ci_low,
            "trough CI high": timing.trough_timing_concentration_ci_high,
            "trough Kuiper p": timing.trough_timing_uniformity_p,
            "timing years": timing.n_timing_years,
        }
    ]
)
timing_summary.round(4)


In [ ]:
print("AOI:", offline_result.aoi_context.display_name)
for message in offline_result.warnings:
    print("warning:", message)


In [ ]:
artifacts = {
    "html": offline_result.artifacts.html,
    "monthly": offline_result.artifacts.monthly_csv,
    "hydrological years": offline_result.artifacts.hydro_years_csv,
    "wet events": offline_result.artifacts.wet_event_csv,
    "low spells": offline_result.artifacts.low_spells_csv,
}
for label, path in artifacts.items():
    assert path.exists(), (label, path)
    print(f"{label}: {path}")

monthly_columns = set(pd.read_csv(artifacts["monthly"], nrows=0).columns)
assert {"date", "extent_pct", "regime", "route"} <= monthly_columns
assert set(pd.read_csv(artifacts["hydrological years"], nrows=0).columns) >= {
    "hy_year",
    "boundary_basis",
    "regime",
    "route",
}
display(FileLink(artifacts["html"]))


In [ ]:
cli_output = OUTPUT_DIR / "cli-fitzroy"
completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "hydroseason",
        "run",
        "--water-source",
        str(CASE_STUDY_CSV),
        "--output-dir",
        str(cli_output),
        "--aoi-name",
        "Fitzroy River (WA) CLI",
        "--no-progress",
        "--json",
    ],
    cwd=PROJECT_ROOT / "notebooks",
    text=True,
    capture_output=True,
    check=True,
)
cli_summary = json.loads(completed.stdout)
assert Path(cli_summary["output_dir"]) == cli_output.resolve()
assert Path(cli_summary["html"]).exists()
cli_summary


In [ ]:
dates = pd.date_range("2010-01-01", periods=12 * 7, freq="MS")
values = []
for _year in range(7):
    values.extend([8.0] * 11 + [0.0])
for year, month in enumerate([1, 3, 5, 7, 9, 11, 1]):
    values[12 * year + month - 1] = 8.1
short_record = pd.DataFrame(
    {"extent_pct": values, "invalid_pct": 0.0},
    index=dates,
)
short_timing = analyze_catchment(
    short_record,
    n_bootstrap=999,
    random_state=0,
).regime
assert short_timing.regime == "marginal"
assert short_timing.n_timing_years == 7
assert any("little power" in caveat for caveat in short_timing.caveats)
pd.DataFrame(
    [
        {
            "regime": short_timing.regime,
            "timing years": short_timing.n_timing_years,
            "Kuiper p": short_timing.peak_timing_uniformity_p,
            "caveats": " ".join(short_timing.caveats),
        }
    ]
)


## Optional live DEA validation

The remaining acquisition checks are disabled unless `HYDROSEASON_RUN_LIVE_DEA=1`. They first run `hydroseason doctor`, then use committed small AOIs for a real single run and two-row batch. These paths require a supported DEA environment and network access; no outcome is fabricated when they are skipped.

In [ ]:
if RUN_LIVE_DEA:
    doctor = subprocess.run(
        [sys.executable, "-m", "hydroseason", "doctor"],
        cwd=PROJECT_ROOT / "notebooks",
        text=True,
        capture_output=True,
        check=False,
    )
    print(doctor.stdout)
    if doctor.returncode:
        raise RuntimeError(doctor.stderr or "hydroseason doctor reported an unsupported environment")


In [ ]:
if RUN_LIVE_DEA:
    live_result = run_hydroseason(
        None,
        output_dir=OUTPUT_DIR / "live-single",
        cache_dir=OUTPUT_DIR / "cache/live-single",
        aoi=SMALL_AOI,
        aoi_name="Fitzroy/Kimberley live sample",
        start_date="2024-01-01",
        end_date="2024-12-01",
        progress=True,
        show_map="auto",
    )
    print(live_result.warnings)
else:
    print("Live DEA single-AOI run skipped; set HYDROSEASON_RUN_LIVE_DEA=1 to enable it.")


In [ ]:
if RUN_LIVE_DEA:
    first = load_aoi(SMALL_AOI).assign(aoi_id="fitzroy-kimberley")
    second = load_aoi(SECOND_AOI).to_crs(first.crs).assign(aoi_id="gilbert-river")
    batch_aois = gpd.GeoDataFrame(
        pd.concat([first, second], ignore_index=True),
        geometry="geometry",
        crs=first.crs,
    )
    batch = run_hydroseason_many(
        batch_aois,
        output_dir=OUTPUT_DIR / "live-batch",
        cache_dir=OUTPUT_DIR / "cache/live-batch",
        start_date="2024-01-01",
        end_date="2024-12-01",
        id_col="aoi_id",
        workers="auto",
        progress=True,
        show_map="auto",
    )
    for outcome in batch.outcomes:
        print(outcome.id, outcome.succeeded, outcome.error_type)
    batch.raise_for_failures()
else:
    print("Live DEA batch skipped; no fabricated batch result is shown.")


In [ ]:
tested = ["environment/API preflight", "offline Python workflow", "report + CSV bundle", "CLI JSON workflow", "short-record guard"]
if RUN_LIVE_DEA:
    tested.extend(["live DEA single AOI", "live DEA two-row batch"])
print("Tested:", ", ".join(tested))
print(
    "Not tested in this run: "
    + ("nothing from the configured routine" if RUN_LIVE_DEA else "live DEA acquisition and batch scheduling")
)
